# Superstore Sales Analysis (Week 2 Day 5 Mini Project)

This notebook analyzes the US Superstore dataset to understand state/city performance, customer concentration (Pareto principle), and to propose an actionable marketing strategy.

## 1. Load the dataset

Load the Superstore data into a pandas DataFrame. Update the path if your file is stored elsewhere.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Try common locations (Colab upload, local workspace, or files/ folder)
data_paths = [
    "files/US Superstore data.xls",
    "US Superstore data.xls",
    "/content/US Superstore data.xls",
    "/content/USSuperstoredata.xls",
    "USSuperstoredata.xls",
]
market_df = None
for path in data_paths:
    try:
        market_df = pd.read_excel(path)
        print(f"Loaded dataset from: {path}")
        break
    except FileNotFoundError:
        continue

if market_df is None:
    raise FileNotFoundError(
        "Superstore dataset was not found. Please place the Excel file in the working directory "
        "or in a files/ subfolder, or update data_paths."
    )

print("Shape:", market_df.shape)
market_df.head()

## 2. Preprocess the data

Before analysis, check for missing values and duplicates, and parse date columns (`Order Date`, `Ship Date`) as datetime types.

In [ ]:
# Parse date columns
date_columns = ["Order Date", "Ship Date"]
for col in date_columns:
    if col in market_df.columns:
        market_df[col] = pd.to_datetime(market_df[col], errors="coerce")

# Data quality checks
print("Missing values per column:")
print(market_df.isna().sum())
print("\nDuplicate rows:", market_df.duplicated().sum())

# Optional: inspect parsed date dtypes
print("\nDate column dtypes:")
print(market_df[date_columns].dtypes)

# If duplicates should be removed, uncomment the next line after review:
# market_df = market_df.drop_duplicates().reset_index(drop=True)

## 3. Dataset overview

Review structure and key columns used in the analysis.

In [ ]:
market_df.info()

## 4. State sales and profit analysis

Compare total sales and total profit by state to identify top performers and mismatches between revenue and profitability.

In [ ]:
state_sales = market_df.groupby("State")["Sales"].sum().sort_values(ascending=False)
state_profit = market_df.groupby("State")["Profit"].sum().sort_values(ascending=False)
state_comparison = market_df.groupby("State")[["Sales", "Profit"]].sum().sort_values("Sales", ascending=False)

print("Top 10 states by sales:")
print(state_sales.head(10))
print("\nTop 10 states by profit:")
print(state_profit.head(10))
print("\nSales and profit comparison (top 10 by sales):")
print(state_comparison.head(10))

### Interpretation: sales vs profit

Compare high-sales states directly. For example, states like California and New York often show that higher sales do not always translate proportionally to higher profit, so profitability should be evaluated alongside revenue.

In [ ]:
focus_states = [state for state in ["California", "New York"] if state in state_comparison.index]
if focus_states:
    print(state_comparison.loc[focus_states])
else:
    print("California/New York not found in the dataset; showing top 2 states by sales instead.")
    print(state_comparison.head(2))

## 5. State profitability differences

Identify states with the highest and lowest total profit to assess variation in profitability.

In [ ]:
print("States by total profit (descending):")
print(state_profit)
print("\nLowest-profit states:")
print(state_profit.sort_values().head(10))

## 6. City sales and profit analysis

Analyze sales and profit by city to find where revenue is strong but profit is weak (or negative).

In [ ]:
city_sales = market_df.groupby("City")["Sales"].sum().sort_values(ascending=False)
city_profit = market_df.groupby("City")["Profit"].sum().sort_values(ascending=False)

city_sales_profit_comparison = pd.DataFrame({
    "Sales": city_sales,
    "Profit": city_profit
}).sort_values("Sales", ascending=False)

print("Top 20 cities by sales (with profit):")
print(city_sales_profit_comparison.head(20))
print("\nTop 20 cities by profit:")
print(city_profit.head(20).to_frame("Profit"))

### Cities where high sales do not imply high profit

Review cities with strong sales but low/negative profit (for example, Philadelphia is a common case in the Superstore dataset) to identify pricing, discount, or cost issues.

In [ ]:
high_sales_low_profit_cities = city_sales_profit_comparison[
    (city_sales_profit_comparison["Sales"] >= city_sales_profit_comparison["Sales"].median()) &
    (city_sales_profit_comparison["Profit"] <= 0)
].sort_values(["Profit", "Sales"], ascending=[True, False])

print("High-sales, non-positive-profit cities:")
print(high_sales_low_profit_cities.head(20))

## 7. Pareto principle: do the top 20% of unique customers generate ~80% of profit?

Aggregate profit by **unique customer** (not rows/orders), then select the top 20% of unique customers by total profit and compare their profit contribution to overall profit.

In [ ]:
customer_profit = (
    market_df.groupby("Customer Name", as_index=False)["Profit"]
    .sum()
    .sort_values("Profit", ascending=False)
    .reset_index(drop=True)
)

unique_customer_count = market_df["Customer Name"].nunique()
top_20_percent_customer_count = int(np.ceil(unique_customer_count * 0.2))

top_profit_customers = customer_profit.head(top_20_percent_customer_count)
top_profit_customers_profit_total = top_profit_customers["Profit"].sum()
total_profit = market_df["Profit"].sum()
top_profit_customers_profit_share = (top_profit_customers_profit_total / total_profit * 100) if total_profit != 0 else np.nan

print(f"Unique customers: {unique_customer_count}")
print(f"Top 20% of unique customers (count): {top_20_percent_customer_count}")
print(f"Profit from top 20% unique customers: {top_profit_customers_profit_total:.2f}")
print(f"Total profit: {total_profit:.2f}")
print(f"Profit share from top 20% unique customers: {top_profit_customers_profit_share:.2f}%")

pareto_holds_profit = pd.notna(top_profit_customers_profit_share) and top_profit_customers_profit_share >= 80
print(f"Pareto principle (~80% profit from top 20% customers) holds: {pareto_holds_profit}")

## 8. Cumulative sales curve by unique customers

Aggregate sales by **unique customer** first, sort customers by sales descending, then compute cumulative sales percentage across customer ranks.

In [ ]:
customer_sales = (
    market_df.groupby("Customer Name")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

cumulative_sales_percent = customer_sales.cumsum() / customer_sales.sum() * 100
customer_rank = np.arange(1, len(cumulative_sales_percent) + 1)

plt.figure(figsize=(10, 6))
plt.plot(customer_rank, cumulative_sales_percent.values)
plt.xlabel("Number of unique customers (ranked by sales)")
plt.ylabel("Cumulative sales (%)")
plt.title("Cumulative Sales by Unique Customer")
plt.grid(True)
plt.tight_layout()
plt.show()

# Reference point: sales contribution of top 20% unique customers
top_20_percent_sales_share = cumulative_sales_percent.iloc[top_20_percent_customer_count - 1]
print(f"Cumulative sales share at top 20% unique customers (rank {top_20_percent_customer_count}): {top_20_percent_sales_share:.2f}%")

## 9. Top 20 customers by sales

After evaluating customer concentration (Pareto and cumulative curve), list the top 20 unique customers by total sales.

In [ ]:
top_20_customers_by_sales = customer_sales.head(20)
print("Top 20 unique customers by sales:")
print(top_20_customers_by_sales.to_frame("Sales"))

## 10. Moderate-performing states to target

Identify moderate-performing states (middle of the sales ranking) as candidates for growth-focused marketing where the business has traction but room to expand.

In [ ]:
# Define "moderate-performing" states as ranks 11-20 by total sales (adjustable)
moderate_sales_states = state_sales.sort_values(ascending=False).iloc[10:20].to_frame("Sales")
moderate_sales_states["Profit"] = state_profit.reindex(moderate_sales_states.index).values

print("Moderate-performing states (sales ranks 11-20):")
print(moderate_sales_states)

# For context: high performers and low performers
high_performing_states = state_comparison.head(10)
low_performing_states = state_comparison.sort_values("Sales", ascending=True).head(10)
print("\nTop-performing states by sales (context):")
print(high_performing_states)
print("\nLowest-sales states (context):")
print(low_performing_states)

## 11. Marketing strategy recommendation

**Actionable recommendations based on the analysis:**
- **Prioritize moderate-performing states for growth:** Focus campaigns, promotions, and sales enablement on the moderate-performing states identified above (sales ranks 11-20: North Carolina, Indiana, Georgia, Kentucky, New Jersey, Arizona, Wisconsin, Colorado, Tennessee, and Minnesota), where there is existing demand and room to increase market share without the saturation of top states.
- **Protect and optimize top markets:** Maintain presence in top sales/profit states (see the top-performing states identified above), but optimize spend efficiency rather than assuming growth requires proportional budget increases.
- **Investigate high-sales, low-profit cities:** In cities with strong sales but weak/negative profit (see the high-sales, low-profit cities identified above, e.g. Philadelphia when present), diagnose discounting, shipping costs, and product mix before increasing marketing investment.
- **Address underperforming states selectively:** For low-sales/low-profit states (see the lowest-sales states identified above), run targeted experiments to identify root causes (pricing, logistics, assortment, awareness) and scale only where unit economics improve.
- **Leverage customer concentration insights:** Use the Pareto and cumulative sales findings to balance retention of high-value unique customers (top segment by profit/sales) with acquisition in the targeted moderate-performing states.